# Exploration des événements sismiques EMSC

Ce notebook explore la réponse JSON de l'API FDSN Event de SeismicPortal. Il sert à observer les données avant de choisir les colonnes du futur modèle PostgreSQL/PostGIS.

## Objectifs

- interroger un échantillon historique ;
- inspecter la structure de la réponse ;
- mesurer les valeurs manquantes et les doublons ;
- identifier les champs utiles pour la magnitude, la localisation et le temps ;
- conserver les observations nécessaires avant la conception du schéma.

In [1]:
import pandas as pd
import requests

API_URL = "https://www.seismicportal.eu/fdsnws/event/1/query"
HISTORY_START = "2026-01-01T00:00:00Z"
REQUEST_TIMEOUT_SECONDS = 30

print(f"Source : {API_URL}")
print(f"Historique depuis : {HISTORY_START}")

Source : https://www.seismicportal.eu/fdsnws/event/1/query
Historique depuis : 2026-01-01T00:00:00Z


In [2]:
from datetime import datetime, timezone, timedelta
from typing import Any

HISTORY_END = datetime.now(timezone.utc)
# Créer une liste de dates par pas de 1 jour
date_list = pd.date_range(start=HISTORY_START, end=HISTORY_END, freq='D').tolist()

all_responses = pd.DataFrame()

for d in date_list:
    d_plus_one = datetime(d.year, d.month, d.day, tzinfo=timezone.utc) + timedelta(days=1)
    params: dict[str, Any] = {
        "limit": 10000,
        "start": d,
        "end": d_plus_one,
        "format": "json",
        "nodata": 204,
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=REQUEST_TIMEOUT_SECONDS,
    ).json()["features"]
    records = response
    events = pd.json_normalize(records)
    all_responses = pd.concat([all_responses, events], ignore_index=True)

In [3]:
# Enregistrement des réponses dans un fichier parquet
all_responses.to_parquet("all_responses.parquet", index=False, engine="pyarrow")

## Suite de l'exploration

Après exécution, refaire l'analyse avec une période historique plus large et une fenêtre récente. Documenter les identifiants stables, les champs de magnitude, les coordonnées, les pays et les marqueurs de mise à jour avant de définir les tables PostgreSQL/PostGIS.

In [4]:
from pydantic import BaseModel

class Geometri(BaseModel):
    type: str
    coordinates: list[float]

class Properties(BaseModel):
    source_id: int
    source_catalog: str
    last_update: datetime
    time: datetime
    flynn_region: str
    lat: float
    lon: float
    depth: float
    evtype: str
    auth: str
    mag: float
    magtype: str
    unid: str

class Feature(BaseModel):
    type: str
    id: str
    geometry: Geometri
    properties: Properties